# Sumarização por Ramo — Bases Anonimizadas SUSEP

Gera **3 tabelas** (uma por ramo), cada uma com merge R_ ↔ S_ seguido de agrupamento geográfico:

| Ramo | Merge R↔S por | Agrupamento | Colunas de prêmio | Colunas de sinistro |
|------|---------------|-------------|--------------------|---------|
| **auto** | `cod_apo, cod_endosso, item, regiao` | `cep_per` | `premio_total_auto` | `indeniz` |
| **comp** | `cod_apo, cod_endosso, tipo, classe, item, cobertura, uf` | `uf` | `premio` | `indeniz` |
| **rural** | `cod_apo, cod_endosso, cod_item, cobertura, cob_fundo, cod_mod, cultura, munic, uf, id_bem` | `munic` (+`uf`) | `premio` | `indeniz`, `desp_sin` |

### Abordagem
1. Para cada ramo, filtrar os datasets harmonizados de riscos e sinistros
2. Agregar prêmios e sinistros separadamente pela chave geográfica
3. Fazer merge dos agregados (outer join: todos os locais com prêmio e/ou sinistro)
4. Calcular sinistralidade

### Engine
- **Polars lazy** (`scan_parquet` → `filter` → `group_by` → `agg` → `collect`) — nunca materializa os ~235M linhas
- **Pandas** apenas para exibição das tabelas finais já agregadas (poucos milhares de linhas)

## 1 — Setup e Configuração

In [1]:
import polars as pl
from pathlib import Path

UNIFIED_ROOT = Path("data/unified")
RISCOS_DIR = UNIFIED_ROOT / "parquet" / "harmonizado_riscos"
SINISTROS_DIR = UNIFIED_ROOT / "parquet" / "harmonizado_sinistros"
OUT_DIR = UNIFIED_ROOT / "csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Chaves de integração R↔S (conforme PLANO seção E.3) ────────────────
CHAVES_MERGE = {
    "auto":  ["cod_apo", "cod_endosso", "item", "regiao"],
    "comp":  ["cod_apo", "cod_endosso", "tipo", "classe", "item", "cobertura", "uf"],
    "rural": ["cod_apo", "cod_endosso", "cod_item", "cobertura", "cob_fundo",
              "cod_mod", "cultura", "munic", "uf", "id_bem"],
}

# ── Coluna geográfica de agrupamento por ramo ──────────────────────────
GEO_COL = {
    "auto":  "cep_per",     # CEP de pernoite do veículo (riscos)
    "comp":  "uf",          # UF do risco
    "rural": "munic",       # Código do município
}

# Nos sinistros a coluna pode ter nome diferente → renomear para GEO_COL
GEO_COL_SIN = {
    "auto":  "cep",         # S_AUTO tem 'cep' em vez de 'cep_per'
    "comp":  "uf",
    "rural": "munic",
}

# ── Coluna(s) extra de contexto geográfico na saída ───────────────────
GEO_EXTRA = {
    "auto":  [],
    "comp":  [],
    "rural": ["uf"],
}

# ── Coluna de prêmio por ramo ─────────────────────────────────────────
PREMIO_COL = {
    "auto":  "premio_total_auto",
    "comp":  "premio",
    "rural": "premio",
}

# ── Colunas de sinistro por ramo ──────────────────────────────────────
SINISTRO_COLS = {
    "auto":  ["indeniz"],
    "comp":  ["indeniz"],
    "rural": ["indeniz", "desp_sin"],
}

print(f"✓ Setup — Polars {pl.__version__}")
print(f"  Riscos:    {RISCOS_DIR}")
print(f"  Sinistros: {SINISTROS_DIR}")
for ramo in ["auto", "comp", "rural"]:
    print(f"  {ramo:6}: merge por {CHAVES_MERGE[ramo]} → agrupar por {GEO_COL[ramo]}")

✓ Setup — Polars 1.39.0
  Riscos:    data/unified/parquet/harmonizado_riscos
  Sinistros: data/unified/parquet/harmonizado_sinistros
  auto  : merge por ['cod_apo', 'cod_endosso', 'item', 'regiao'] → agrupar por cep_per
  comp  : merge por ['cod_apo', 'cod_endosso', 'tipo', 'classe', 'item', 'cobertura', 'uf'] → agrupar por uf
  rural : merge por ['cod_apo', 'cod_endosso', 'cod_item', 'cobertura', 'cob_fundo', 'cod_mod', 'cultura', 'munic', 'uf', 'id_bem'] → agrupar por munic


## 2 — Funções de Agregação com Polars Lazy

Funções reutilizáveis que usam `pl.scan_parquet()` (lazy) para filtrar por ramo
e agregar pela chave geográfica sem materializar os dados completos em RAM.

In [2]:
def _glob_ramo(base_dir: Path, ramo: str, prefixo: str) -> str:
    """Retorna glob para arquivos do ramo: ex. R_AUTO_*.parquet"""
    tag = ramo.upper()
    return str(base_dir / f"{prefixo}_{tag}_*.parquet")


def agregar_premios(ramo: str) -> pl.LazyFrame:
    """Lazy scan de harmonizado_riscos (só arquivos do ramo) → group_by geográfico → soma prêmio."""
    geo = GEO_COL[ramo]
    extras = GEO_EXTRA[ramo]
    premio_col = PREMIO_COL[ramo]
    group_cols = [geo] + extras

    lf = (
        pl.scan_parquet(_glob_ramo(RISCOS_DIR, ramo, "R"))
        .filter(pl.col(geo).is_not_null())
        .group_by(group_cols)
        .agg(
            pl.col(premio_col).sum().alias("premio"),
            pl.col(premio_col).count().alias("n_apolices"),
        )
    )
    return lf


def agregar_sinistros(ramo: str) -> pl.LazyFrame:
    """Lazy scan de harmonizado_sinistros → join com riscos (via CHAVES_MERGE)
    para herdar a variável geográfica do risco → group_by geográfico → soma sinistro.

    Mesma lógica para todos os ramos:
      auto:  join herda cep_per (não existe em S_AUTO)
      comp:  join por chaves que já incluem uf
      rural: join por chaves que já incluem munic + uf
    """
    geo = GEO_COL[ramo]
    extras = GEO_EXTRA[ramo]
    sin_cols = SINISTRO_COLS[ramo]
    group_cols = [geo] + extras
    merge_keys = CHAVES_MERGE[ramo]

    agg_exprs = [
        pl.col(c).sum().alias(c) for c in sin_cols
    ] + [
        pl.len().alias("n_sinistros"),
    ]

    lf_sin = pl.scan_parquet(_glob_ramo(SINISTROS_DIR, ramo, "S"))

    # Colunas geográficas que precisam vir dos riscos (não estão nas chaves de merge)
    geo_extras_risco = [c for c in [geo] + extras if c not in merge_keys]
    lf_risco = (
        pl.scan_parquet(_glob_ramo(RISCOS_DIR, ramo, "R"))
        .select(merge_keys + geo_extras_risco)
    )

    lf = (
        lf_sin
        .join(lf_risco, on=merge_keys, how="left")
        .filter(pl.col(geo).is_not_null())
        .group_by(group_cols)
        .agg(agg_exprs)
    )
    return lf


def merge_e_agrupar(ramo: str) -> pl.DataFrame:
    """Agrega prêmios e sinistros via Polars lazy, faz join pelo campo geográfico."""
    geo = GEO_COL[ramo]
    extras = GEO_EXTRA[ramo]
    join_keys = [geo] + extras

    print(f"\n{'='*60}")
    print(f"RAMO: {ramo.upper()} — agrupar por '{geo}'")
    print(f"  Chaves merge R↔S: {CHAVES_MERGE[ramo]}")
    print(f"{'='*60}")

    print(f"  Agregando prêmios...")
    lf_premio = agregar_premios(ramo)
    df_premio = lf_premio.collect()
    print(f"  Riscos {ramo}: → {len(df_premio):,} chaves geográficas")

    print(f"  Agregando sinistros...")
    lf_sinistro = agregar_sinistros(ramo)
    df_sinistro = lf_sinistro.collect()
    print(f"  Sinistros {ramo}: → {len(df_sinistro):,} chaves geográficas")

    # Join outer prêmio ↔ sinistro
    if len(df_premio) == 0 and len(df_sinistro) == 0:
        print(f"  ⚠ Nenhum dado para ramo {ramo}")
        return pl.DataFrame()

    if len(df_sinistro) == 0:
        df = df_premio.with_columns(
            *[pl.lit(0.0).alias(c) for c in SINISTRO_COLS[ramo]],
            pl.lit(0).cast(pl.UInt32).alias("n_sinistros"),
        )
    elif len(df_premio) == 0:
        df = df_sinistro.with_columns(
            pl.lit(0.0).alias("premio"),
            pl.lit(0).cast(pl.UInt32).alias("n_apolices"),
        )
    else:
        df = df_premio.join(df_sinistro, on=join_keys, how="full", coalesce=True)
        # Preencher nulls do join
        df = df.with_columns(
            pl.col("premio").fill_null(0.0),
            pl.col("n_apolices").fill_null(0),
            *[pl.col(c).fill_null(0.0) for c in SINISTRO_COLS[ramo]],
            pl.col("n_sinistros").fill_null(0),
        )

    # Sinistro total e sinistralidade
    sin_total_expr = sum(pl.col(c) for c in SINISTRO_COLS[ramo])
    df = df.with_columns(
        sin_total_expr.alias("sinistro_total"),
    ).with_columns(
        pl.when(pl.col("premio") > 0)
        .then((pl.col("sinistro_total") / pl.col("premio")).round(4))
        .otherwise(None)
        .alias("sinistralidade"),
    ).sort(geo)

    total_p = df["premio"].sum()
    total_s = df["sinistro_total"].sum()

    print(f"\n  ✓ Tabela final: {len(df):,} linhas | colunas: {df.columns}")
    print(f"    Prêmio total:   R$ {total_p:>18,.2f}")
    print(f"    Sinistro total: R$ {total_s:>18,.2f}")
    if total_p > 0:
        print(f"    Sinistralidade: {total_s / total_p:.2%}")

    return df

print("✓ Funções de agregação Polars definidas")

✓ Funções de agregação Polars definidas


## 3 — Tabela AUTO: Merge R↔S → Agrupamento por CEP de Pernoite

- **Merge** por: `cod_apo`, `cod_endosso`, `item`, `regiao`
- **Agrupamento** por: `cep_per` (CEP de pernoite do veículo)
- **Prêmio**: `premio_total_auto` (soma das coberturas pre_*)
- **Sinistro**: `indeniz`

In [3]:
df_auto = merge_e_agrupar("auto")
df_auto.head(20).to_pandas()


RAMO: AUTO — agrupar por 'cep_per'
  Chaves merge R↔S: ['cod_apo', 'cod_endosso', 'item', 'regiao']
  Agregando prêmios...
  Riscos auto: → 750,338 chaves geográficas
  Agregando sinistros...
  Sinistros auto: → 340,098 chaves geográficas

  ✓ Tabela final: 750,338 linhas | colunas: ['cep_per', 'premio', 'n_apolices', 'indeniz', 'n_sinistros', 'sinistro_total', 'sinistralidade']
    Prêmio total:   R$  84,460,353,592.00
    Sinistro total: R$  24,992,488,902.51
    Sinistralidade: 29.59%


,cep_per,premio,n_apolices,indeniz,n_sinistros,sinistro_total,sinistralidade
0,00000000,224910271.0,192831,47265970.0,15913,47265970.0,0.2102
1,00000002,19.0,1,0.0,0,0.0,0.0000
2,00000004,319.0,1,0.0,0,0.0,0.0000
3,00372012,2195.0,1,0.0,0,0.0,0.0000
4,00436000,141.0,1,221.0,1,221.0,1.5674
5,00730020,7344.0,2,0.0,0,0.0,0.0000
6,00779154,1768.0,1,0.0,0,0.0,0.0000
7,01000000,131309304.0,443545,93980418.0,11638,93980418.0,0.7157
8,01000001,21983068.0,15975,2482605.0,293,2482605.0,0.1129
9,0100001,3655.0,1,0.0,0,0.0,0.0000


In [9]:
# Validação: apólices com cep_per < 8 dígitos
df_cep_curto = df_auto.filter(pl.col("cep_per").str.len_chars() < 8)
n_chaves_curtas = len(df_cep_curto)
n_apolices_curtas = df_cep_curto["n_apolices"].sum() if n_chaves_curtas > 0 else 0
n_apolices_total = df_auto["n_apolices"].sum()
pct = (n_apolices_curtas / n_apolices_total * 100) if n_apolices_total > 0 else 0

print(f"Validação: cep_per com menos de 8 dígitos")
print(f"  Chaves geográficas afetadas: {n_chaves_curtas:,} de {len(df_auto):,}")
print(f"  Apólices afetadas:           {n_apolices_curtas:,} de {n_apolices_total:,} ({pct:.2f}%)")

if n_chaves_curtas > 0:
    print(f"\n  Distribuição por tamanho do cep_per:")
    dist = (
        df_auto
        .with_columns(pl.col("cep_per").str.len_chars().alias("len_cep"))
        .group_by("len_cep")
        .agg(
            pl.len().alias("n_chaves"),
            pl.col("n_apolices").sum().alias("n_apolices"),
        )
        .sort("len_cep")
    )
    print(dist.to_pandas().to_string(index=False))

Validação: cep_per com menos de 8 dígitos
  Chaves geográficas afetadas: 2 de 750,338
  Apólices afetadas:           2 de 64,981,335 (0.00%)

  Distribuição por tamanho do cep_per:
 len_cep  n_chaves  n_apolices
       7         2           2
       8    750336    64981333


In [10]:
# Análise: impacto do cep_per "00000000" na base AUTO
row_zeros = df_auto.filter(pl.col("cep_per") == "00000000")

if len(row_zeros) == 0:
    print("cep_per '00000000' não encontrado na base AUTO.")
else:
    apo_zeros = row_zeros["n_apolices"][0]
    premio_zeros = row_zeros["premio"][0]
    apo_total = df_auto["n_apolices"].sum()
    premio_total = df_auto["premio"].sum()

    print("Impacto do cep_per '00000000' na base AUTO:")
    print(f"  Apólices:  {apo_zeros:>12,} de {apo_total:>12,}  → {apo_zeros / apo_total * 100:.2f}%")
    print(f"  Prêmio: R$ {premio_zeros:>18,.2f} de R$ {premio_total:>18,.2f}  → {premio_zeros / premio_total * 100:.2f}%")

Impacto do cep_per '00000000' na base AUTO:
  Apólices:       192,831 de   64,981,335  → 0.30%
  Prêmio: R$     224,910,271.00 de R$  84,460,353,592.00  → 0.27%


## 4 — Tabela COMP: Merge R↔S → Agrupamento por UF

- **Merge** por: `cod_apo`, `cod_endosso`, `tipo`, `classe`, `item`, `cobertura`, `uf`
- **Agrupamento** por: `uf`
- **Prêmio**: `premio`
- **Sinistro**: `indeniz`

In [8]:
df_comp = merge_e_agrupar("comp")
df_comp.head(46).to_pandas()


RAMO: COMP — agrupar por 'uf'
  Chaves merge R↔S: ['cod_apo', 'cod_endosso', 'tipo', 'classe', 'item', 'cobertura', 'uf']
  Agregando prêmios...
  Riscos comp: → 45 chaves geográficas
  Agregando sinistros...
  Sinistros comp: → 28 chaves geográficas

  ✓ Tabela final: 45 linhas | colunas: ['uf', 'premio', 'n_apolices', 'indeniz', 'n_sinistros', 'sinistro_total', 'sinistralidade']
    Prêmio total:   R$  11,780,498,468.00
    Sinistro total: R$  23,012,013,966.00
    Sinistralidade: 195.34%


,uf,premio,n_apolices,indeniz,n_sinistros,sinistro_total,sinistralidade
0,,1.080000e+02,16,1.000000e+00,1,1.000000e+00,0.0093
1,AC,1.390800e+07,165125,2.487966e+06,465,2.487966e+06,0.1789
2,AL,4.705116e+07,679228,1.939362e+07,6669,1.939362e+07,0.4122
3,AM,8.407000e+07,543625,8.398736e+06,2036,8.398736e+06,0.0999
4,AP,1.277793e+07,133484,6.460136e+06,1922,6.460136e+06,0.5056
5,BA,2.593392e+08,3326149,1.250548e+09,148561,1.250548e+09,4.8221
6,CE,1.540850e+08,1752237,1.994572e+07,5726,1.994572e+07,0.1294
7,DF,2.356250e+08,2728082,4.009593e+07,19783,4.009593e+07,0.1702
8,ES,1.503671e+08,1773137,3.497915e+07,17612,3.497915e+07,0.2326
9,G,1.120000e+02,1,0.000000e+00,0,0.000000e+00,0.0000


## 5 — Tabela RURAL: Merge R↔S → Agrupamento por Município

- **Merge** por: `cod_apo`, `cod_endosso`, `cod_item`, `cobertura`, `cob_fundo`, `cod_mod`, `cultura`, `munic`, `uf`, `id_bem`
- **Agrupamento** por: `munic` (+`uf`)
- **Prêmio**: `premio`
- **Sinistro**: `indeniz` + `desp_sin`

In [6]:
df_rural = merge_e_agrupar("rural")
df_rural.head(20).to_pandas()


RAMO: RURAL — agrupar por 'munic'
  Chaves merge R↔S: ['cod_apo', 'cod_endosso', 'cod_item', 'cobertura', 'cob_fundo', 'cod_mod', 'cultura', 'munic', 'uf', 'id_bem']
  Agregando prêmios...
  Riscos rural: → 6,058 chaves geográficas
  Agregando sinistros...
  Sinistros rural: → 3,262 chaves geográficas

  ✓ Tabela final: 6,062 linhas | colunas: ['munic', 'uf', 'premio', 'n_apolices', 'indeniz', 'desp_sin', 'n_sinistros', 'sinistro_total', 'sinistralidade']
    Prêmio total:   R$  13,433,549,713.77
    Sinistro total: R$   7,601,978,612.72
    Sinistralidade: 56.59%


,munic,uf,premio,n_apolices,indeniz,desp_sin,n_sinistros,sinistro_total,sinistralidade
0,0.0,PI,1766.28,6,0.00,0.00,0,0.00,0.0000
1,0.0,MT,1144.50,2,0.00,0.00,0,0.00,0.0000
2,0.0,MG,396047.13,1252,0.00,0.00,2,0.00,0.0000
3,0.0,RJ,1813.82,10,0.00,0.00,0,0.00,0.0000
4,0.0,MS,1352922.07,204,207638.65,2726.30,12,210364.95,0.1555
5,0.0,PR,4531075.39,1457,4755349.57,28710.24,142,4784059.81,1.0558
6,0.0,,0.00,172,0.00,0.00,0,0.00,NaN
7,0.0,CE,2724.34,12,0.00,0.00,0,0.00,0.0000
8,0.0,SP,3519126.50,1212,2811421.63,9283.04,29,2820704.67,0.8015
9,0.0,RO,61827.09,61,0.00,0.00,0,0.00,0.0000


In [12]:
# Análise: impacto de munic == "0" ou uf vazia na base RURAL
apo_total_r = df_rural["n_apolices"].sum()
premio_total_r = df_rural["premio"].sum()

# munic é string com decimais (ex: "431490.0") — cast via Float64
mask_munic_zero = pl.col("munic").cast(pl.Float64, strict=False).fill_null(0.0) == 0.0
mask_uf_vazia = pl.col("uf").is_null() | (pl.col("uf").str.strip_chars() == "")

df_munic0 = df_rural.filter(mask_munic_zero)
df_uf_vazia = df_rural.filter(mask_uf_vazia)
df_ambos = df_rural.filter(mask_munic_zero | mask_uf_vazia)

def _resumo(label, df_filtrado):
    n = len(df_filtrado)
    apo = df_filtrado["n_apolices"].sum() if n > 0 else 0
    pre = df_filtrado["premio"].sum() if n > 0 else 0
    pct_apo = apo / apo_total_r * 100 if apo_total_r > 0 else 0
    pct_pre = pre / premio_total_r * 100 if premio_total_r > 0 else 0
    print(f"  {label}:")
    print(f"    Chaves:   {n:>8,} de {len(df_rural):,}")
    print(f"    Apólices: {apo:>12,} de {apo_total_r:>12,}  → {pct_apo:.2f}%")
    print(f"    Prêmio: R$ {pre:>18,.2f} de R$ {premio_total_r:>18,.2f}  → {pct_pre:.2f}%")

print("Validação RURAL — registros com munic=0 ou UF vazia:")
_resumo("munic = 0", df_munic0)
_resumo("UF vazia/nula", df_uf_vazia)
_resumo("munic=0 OU UF vazia (união)", df_ambos)

Validação RURAL — registros com munic=0 ou UF vazia:
  munic = 0:
    Chaves:         23 de 6,062
    Apólices:        6,267 de   12,773,353  → 0.05%
    Prêmio: R$      22,590,111.84 de R$  13,433,549,713.77  → 0.17%
  UF vazia/nula:
    Chaves:          1 de 6,062
    Apólices:          172 de   12,773,353  → 0.00%
    Prêmio: R$               0.00 de R$  13,433,549,713.77  → 0.00%
  munic=0 OU UF vazia (união):
    Chaves:         23 de 6,062
    Apólices:        6,267 de   12,773,353  → 0.05%
    Prêmio: R$      22,590,111.84 de R$  13,433,549,713.77  → 0.17%


In [14]:
# Top 20 municípios por n_apolices (excluindo munic=0)
top_munic = (
    df_rural
    .filter(pl.col("munic").cast(pl.Float64, strict=False).fill_null(0.0) > 0)
    .sort("n_apolices", descending=True)
    .head(20)
    .with_columns(
        (pl.col("n_apolices") / apo_total_r * 100).round(2).alias("% apolices"),
        (pl.col("premio") / premio_total_r * 100).round(2).alias("% premio"),
    )
    .select(["munic", "uf", "n_apolices", "% apolices", "premio", "% premio", "sinistralidade"])
)

print(f"Top 20 municípios RURAL por n_apolices (total: {apo_total_r:,} apólices)")
top_munic.to_pandas()

Top 20 municípios RURAL por n_apolices (total: 12,773,353 apólices)


,munic,uf,n_apolices,% apolices,premio,% premio,sinistralidade
0,431490.0,RS,1353036,10.59,8.992011e+07,0.67,0.4647
1,21216.0,SC,422201,3.31,4.104273e+07,0.31,0.8098
2,7388.0,SP,369168,2.89,6.038142e+08,4.49,0.2802
3,3681.0,RS,183287,1.43,3.779761e+07,0.28,0.1902
4,24828.0,MG,128136,1.00,1.049780e+08,0.78,0.5136
5,23513.0,PR,111544,0.87,3.614944e+07,0.27,0.1697
6,26242.0,GO,83196,0.65,6.489686e+07,0.48,0.9887
7,29665.0,ES,75888,0.59,1.157398e+07,0.09,1.5953
8,30719.0,RO,73795,0.58,2.886226e+07,0.21,1.2124
9,20011.0,RS,48302,0.38,1.616233e+07,0.12,0.0940


    ## 6 — Exportação

In [7]:
exportados = {}

for ramo, df in [("auto", df_auto), ("comp", df_comp), ("rural", df_rural)]:
    if len(df) == 0:
        print(f"  ⚠ {ramo}: sem dados, pulando exportação")
        continue
    out_file = OUT_DIR / f"sumarizacao_{ramo}_por_{GEO_COL[ramo]}.csv"
    df.write_csv(str(out_file))
    exportados[ramo] = out_file
    print(f"  ✓ {out_file.name:50} ({len(df):>6,} linhas)")

print(f"\n{'='*60}")
print("RESUMO GERAL")
print(f"{'='*60}")
for ramo, df in [("auto", df_auto), ("comp", df_comp), ("rural", df_rural)]:
    if len(df) == 0:
        continue
    geo = GEO_COL[ramo]
    total_p = df["premio"].sum()
    total_s = df["sinistro_total"].sum()
    print(f"\n  {ramo.upper()} (por {geo}):")
    print(f"    {len(df):>6,} chaves geográficas")
    print(f"    Prêmio:       R$ {total_p:>18,.2f}")
    print(f"    Sinistro:     R$ {total_s:>18,.2f}")
    if total_p > 0:
        print(f"    Sinistralidade: {total_s / total_p:.2%}")
    print(f"    Arquivo: {exportados.get(ramo, 'N/A')}")

  ✓ sumarizacao_auto_por_cep_per.csv                   (750,338 linhas)
  ✓ sumarizacao_comp_por_uf.csv                        (    45 linhas)
  ✓ sumarizacao_rural_por_munic.csv                    ( 6,062 linhas)

RESUMO GERAL

  AUTO (por cep_per):
    750,338 chaves geográficas
    Prêmio:       R$  84,460,353,592.00
    Sinistro:     R$  24,992,488,902.51
    Sinistralidade: 29.59%
    Arquivo: data/unified/csv/sumarizacao_auto_por_cep_per.csv

  COMP (por uf):
        45 chaves geográficas
    Prêmio:       R$  11,780,498,468.00
    Sinistro:     R$  23,012,013,966.00
    Sinistralidade: 195.34%
    Arquivo: data/unified/csv/sumarizacao_comp_por_uf.csv

  RURAL (por munic):
     6,062 chaves geográficas
    Prêmio:       R$  13,433,549,713.77
    Sinistro:     R$   7,601,978,612.72
    Sinistralidade: 56.59%
    Arquivo: data/unified/csv/sumarizacao_rural_por_munic.csv
